# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

This lane maps onto a decision FlyRank's reviewers actually face every week: limited time, too many pages, which ones get looked at first? It also has a working reference implementation already sitting in this repo — the starter pipeline (`scripts/01`–`05`) builds a transparent baseline score, trains a model, and exports a ranked review queue with reason codes on exactly this dataset. That gives me a real, honest benchmark from day one (baseline vs. model precision@50, verified in `outputs/model_results.json`) instead of a blank page, and a clear bar to try to beat with a stronger label and validation design over the next 7 weeks. I can confirm or swap this lane by Week 4 if the signal audit says otherwise.

## 2. The question: decision, action, cost of a wrong call

**Question:** Given limited review capacity, which content pages should be reviewed first this cycle for refresh, expansion, protection, pruning, or monitoring?

**Unit of analysis:** one row = one content item (`content_id`) at a snapshot in time — the starter data's trailing-90-day window; in the warehouse this would be a client-defined feature window ending at a chosen decision date.

**Who acts, and what they do:** an SEO content reviewer/strategist working a client account, with a fixed number of review slots per cycle (e.g. the team can realistically look at 20–50 pages a week). They open the top of the ranked queue, read each page's reason codes (stale, declining-with-demand, thin, page-one decay risk, low-CTR), and decide whether to refresh, expand, protect, prune, or just keep monitoring it.

**Cost of a wrong call:**
- *False positive* (a page is ranked high but wasn't actually a problem): burns ~30–60 minutes of a reviewer's limited time, and because capacity is fixed, that slot is time a genuinely declining page did not get.
- *False negative* (a real decline never makes the top of the queue): it keeps losing visibility/clicks silently until the next review cycle — the loss compounds and the page may drop off page 1 before anyone notices.
- Because the binding constraint is reviewer capacity, not raw accuracy, **precision@K** (are the top K picks actually worth reviewing?) is the metric that matches the real decision — not overall classification accuracy.

**Why data/ML helps, not just a fixed rule:** this repo's own starter pipeline already tested a fixed weighted rule (the baseline score) against a learned model on the same signals, client-holdout validated. The rule alone reaches precision@50 = 0.24; a random forest on the same observable signals reaches 0.74 (see the numbers pulled live below). That's a real, measured 3x lift — evidence that the pattern across many tangled signals (impression days, position, freshness, CTR, age) is real but too non-linear for a hand-written weighted sum to capture well, which is exactly when ML earns its place over a plain rule.

In [1]:
import json
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

n_rows = len(df)
n_clients = df["client_id"].nunique()

declining_with_demand = (
    (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)
).sum()

low_ctr_visible_page = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
).sum()

print(f"starter rows: {n_rows:,}  |  clients: {n_clients}")
print(
    f"declining_with_demand: {declining_with_demand:,} rows "
    f"({100 * declining_with_demand / n_rows:.1f}% of the slice) -- real review candidates, not a rare edge case"
)
print(
    f"low_ctr_visible_page: {low_ctr_visible_page:,} rows "
    f"({100 * low_ctr_visible_page / n_rows:.1f}% of the slice) -- visible pages under-capturing clicks for their position"
)

with open("../../outputs/model_results.json") as f:
    results = json.load(f)

baseline_p50 = results["baseline"]["baseline_precision_at_50"]
best_model_name = results["best_model"]["name"]
model_p50 = results["models"][best_model_name]["precision_at_50"]

print(
    f"\nverified in outputs/model_results.json (client-holdout split): "
    f"baseline rule precision@50 = {baseline_p50:.2f}  vs.  {best_model_name} precision@50 = {model_p50:.2f}"
)
print(
    "-> of the top 50 pages the ranked queue would send a reviewer, the model gets "
    f"~{model_p50 * 50:.0f} right versus ~{baseline_p50 * 50:.0f} for the fixed rule."
)

starter rows: 30,000  |  clients: 32
declining_with_demand: 13,152 rows (43.8% of the slice) -- real review candidates, not a rare edge case
low_ctr_visible_page: 9,759 rows (32.5% of the slice) -- visible pages under-capturing clicks for their position

verified in outputs/model_results.json (client-holdout split): baseline rule precision@50 = 0.24  vs.  random_forest precision@50 = 0.74
-> of the top 50 pages the ranked queue would send a reviewer, the model gets ~37 right versus ~12 for the fixed rule.


## 4. Careful words: what I can and can't claim

**What this work can say:**
- *Observed / measured*: numbers like "X% of pages in the starter slice show declining-with-demand signals" — a direct count from the data.
- *Directional / associational*: "pages with these signal combinations are more often flagged as declining" — a pattern, not a mechanism.
- *Decision-support*: a ranked queue that helps a reviewer spend limited time on the most promising candidates first, backed by precision@K measured on held-out clients.

**What this work will never say:**
- That a refresh **caused** a recovery — proving causation needs an experiment (e.g. an A/B test on which pages get refreshed), which this observational data cannot give me.
- That I predicted or reverse-engineered a **Google algorithm factor** — I only have FlyRank's own observable search/engagement signals, never Google's ranking internals.
- That a starter-data result (30,000 rows, 32 clients) is a benchmark for the full ~79M-row warehouse — any number carried forward has to be re-earned there with its own validation.
- That `trend_direction` / `is_declining_label` is a "true" outcome rather than a proxy computed from the current window — it's a beginner label, not a future observed outcome, and I'll say so if I keep using it before I build a stronger future-window label.
- Anything that reveals a client name, domain, URL, or raw query — the dataset is pseudonymized and stays that way in every output.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.